# Reddit thread collection

Harvests post metadata from all 32 team subreddits across the 2021–2026 draft
windows. Output is one JSON file per team-year in `data/raw/reddit_threads/`,
consolidated here into a single `threads` table.

Comment bodies are **not** fetched here — this notebook establishes which
threads exist. Matching threads to specific draft picks happens in
`04_match_threads.ipynb`.

In [76]:
from pathlib import Path
import pandas as pd

# PFR team code -> subreddit name
TEAM_SUBREDDITS = {
    "ARI": "AZCardinals",       "ATL": "falcons",
    "BAL": "ravens",            "BUF": "buffalobills",
    "CAR": "panthers",          "CHI": "CHIBears",
    "CIN": "bengals",           "CLE": "Browns",
    "DAL": "cowboys",           "DEN": "DenverBroncos",
    "DET": "detroitlions",      "GNB": "GreenBayPackers",
    "HOU": "Texans",            "IND": "Colts",
    "JAX": "Jaguars",           "KAN": "KansasCityChiefs",
    "LAC": "Chargers",          "LAR": "LosAngelesRams",
    "LVR": "raiders",           "MIA": "miamidolphins",
    "MIN": "minnesotavikings",  "NOR": "Saints",
    "NWE": "Patriots",          "NYG": "NYGiants",
    "NYJ": "nyjets",            "PHI": "eagles",
    "PIT": "steelers",          "SEA": "Seahawks",
    "SFO": "49ers",             "TAM": "buccaneers",
    "TEN": "Tennesseetitans",   "WAS": "Commanders",
}

# Draft dates shift year to year, so over-collect and let
# player-name matching do the filtering later.
DRAFT_YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
WINDOW_START, WINDOW_END = "04-20", "05-06"

BASE = "https://arctic-shift.photon-reddit.com"
RAW = Path("..") / "data" / "raw" / "reddit_threads"
RAW.mkdir(parents=True, exist_ok=True)

outcomes = pd.read_csv("../data/processed/draft_outcomes_2021_2025.csv")

assert len(TEAM_SUBREDDITS) == 32, f"expected 32, got {len(TEAM_SUBREDDITS)}"
missing = set(outcomes["team"].unique()) - set(TEAM_SUBREDDITS)
assert not missing, f"codes with no subreddit: {missing}"
print("mapping verified against draft data")

mapping verified against draft data


### Handling the Washington subreddit move

Washington was the "Football Team" from 2020–2021 before becoming the
Commanders in 2022. The fanbase moved to a new subreddit rather than renaming
in place, and Reddit does not carry post history across such a move — so
`r/Commanders` returns nothing for the 2021 draft window.

The 2021 threads live in `r/WashingtonNFL` (491 posts, busiest thread 1,139
comments). Rather than complicate the main mapping, a small override table
handles team-years that deviate from it.

In [77]:
# Team-years whose subreddit differs from the current one
SUBREDDIT_OVERRIDES = {
    ("WAS", 2021): "WashingtonNFL",
}

def subreddit_for(code, year):
    """Subreddit for a team-year, honouring any override."""
    return SUBREDDIT_OVERRIDES.get((code, year), TEAM_SUBREDDITS[code])

### Resilient HTTP session

Collection makes several hundred requests over several minutes, so transient
failures — DNS hiccups, dropped connections, server-side rate limiting — are
expected rather than exceptional. The session retries automatically with
exponential backoff so one blip doesn't abort a run.

In [78]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)))

### Paginated collection for reproducibility

The `limit="auto"` parameter returns a variable number of posts depending on
server capacity at request time. Repeated calls for the same subreddit and
window returned 491, 266, and 157 posts — meaning the collected dataset was
not reproducible, and every team-year was silently truncated by an unknown
amount.

Explicit limits are capped at 100 per request, so complete collection requires
pagination: request a fixed page, advance the window start to the newest post
received, and repeat until a partial page signals the end. Posts are keyed by
ID during accumulation so that any overlap at page boundaries is deduplicated
rather than double-counted.

In [79]:
import json
import time
import requests

PAGE_SIZE = 100
MAX_PAGES = 50

def fetch_threads(subreddit, year, session, verbose=False):
    """Every post for one subreddit in one draft window."""
    after = f"{year}-{WINDOW_START}"
    before = f"{year}-{WINDOW_END}"
    seen = {}

    for page in range(MAX_PAGES):
        resp = session.get(
            f"{BASE}/api/posts/search",
            params={
                "subreddit": subreddit,
                "after": after,
                "before": before,
                "limit": PAGE_SIZE,
                "sort": "asc",
                "fields": "id,title,created_utc,num_comments,score,author",
            },
            timeout=60,
        )
        resp.raise_for_status()
        batch = resp.json()["data"]

        if not batch:
            break

        new = sum(1 for p in batch if p["id"] not in seen)
        seen.update({p["id"]: p for p in batch})

        if verbose:
            print(f"  page {page}: {len(batch)} returned, {new} new, {len(seen)} total")

        if len(batch) < PAGE_SIZE or new == 0:
            break

        after = max(p["created_utc"] for p in batch)
        time.sleep(0.5)

    return list(seen.values())

In [80]:
rows = []

for code, sub in TEAM_SUBREDDITS.items():
    for year in DRAFT_YEARS:
        out_path = RAW / f"{code}_{year}.json"

        if out_path.exists():
            posts = json.loads(out_path.read_text())
            status = "cached"
        else:
            try:
                posts = fetch_threads(subreddit_for(code, year), year, session)
                out_path.write_text(json.dumps(posts))
                status = "fetched"
                time.sleep(1.0)
            except Exception as exc:
                print(f"FAILED  {code} {year}: {exc}")
                continue

        rows.append({
            "team": code,
            "subreddit": subreddit_for(code, year),
            "year": year,
            "n_posts": len(posts),
            "total_comments": sum(p.get("num_comments", 0) for p in posts),
        })
        print(f"{status:8} {code} {year}  posts={len(posts):4}")

summary = pd.DataFrame(rows)
print("\ndone:", len(summary), "of", 32 * len(DRAFT_YEARS), "team-years")

cached   ARI 2021  posts= 444
cached   ARI 2022  posts= 262
cached   ARI 2023  posts= 429
cached   ARI 2024  posts= 380
cached   ARI 2025  posts= 183
cached   ARI 2026  posts= 268
cached   ATL 2021  posts= 519
cached   ATL 2022  posts= 464
cached   ATL 2023  posts= 491
cached   ATL 2024  posts=1026
cached   ATL 2025  posts= 565
cached   ATL 2026  posts= 300
cached   BAL 2021  posts= 890
cached   BAL 2022  posts= 817
cached   BAL 2023  posts= 766
cached   BAL 2024  posts= 342
cached   BAL 2025  posts= 310
cached   BAL 2026  posts= 449
cached   BUF 2021  posts= 468
cached   BUF 2022  posts= 584
cached   BUF 2023  posts= 362
cached   BUF 2024  posts= 531
cached   BUF 2025  posts= 421
cached   BUF 2026  posts= 337
cached   CAR 2021  posts= 390
cached   CAR 2022  posts= 336
cached   CAR 2023  posts= 526
cached   CAR 2024  posts= 304
cached   CAR 2025  posts= 422
cached   CAR 2026  posts= 300
cached   CHI 2021  posts=1569
cached   CHI 2022  posts= 682
cached   CHI 2023  posts= 654
cached   C

In [81]:
summary.pivot(index="team", columns="year", values="n_posts")

year,2021,2022,2023,2024,2025,2026
team,,,,,,
ARI,444,262,429,380,183,268
ATL,519,464,491,1026,565,300
BAL,890,817,766,342,310,449
BUF,468,584,362,531,421,337
CAR,390,336,526,304,422,300
CHI,1569,682,654,1103,653,510
CIN,762,426,464,301,234,349
CLE,628,393,223,170,620,383
DAL,496,358,441,314,293,437


### Consolidating harvested threads

Each team-year was saved as its own JSON file to make the collection
resumable. For analysis we want a single flat table: one row per thread,
carrying the team and year it came from.

The team code and year are recovered from the filename (`NOR_2022.json`),
which is why the naming convention was chosen deliberately.

In [82]:
records = []

for path in sorted(RAW.glob("*.json")):
    code, year = path.stem.split("_")
    for p in json.loads(path.read_text()):
        records.append({
            "team": code,
            "year": int(year),
            "post_id": p["id"],
            "title": p.get("title", ""),
            "num_comments": p.get("num_comments", 0),
            "created_utc": p.get("created_utc"),
        })

threads = pd.DataFrame(records)
print("threads:", len(threads))
print("team-years:", threads.groupby(["team", "year"]).ngroups)
threads.head()

threads: 98231
team-years: 192


,team,year,post_id,title,num_comments,created_utc
0,ARI,2021,muf3ao,Mock Draft,2,1618880309
1,ARI,2021,muid0m,No more free agency moves please...,0,1618891936
2,ARI,2021,muiy5f,The state flag uniforms are hideous. There. I ...,32,1618894255
3,ARI,2021,mum0ei,Top CB's going into the draft.,0,1618908196
4,ARI,2021,mus1g3,New Uniforms Heavily Hinted At,40,1618930235
